# BASELINE ADDITIONAL TESTS

TTF-IDF, Jaccard and MiniLM: retrieval runtime, input ingredient counts, synonyms and unknown ingredients

In [1]:
# IMPORTS

import time
import random
import numpy as np
import pandas as pd
from ingredient_normalisation import normalise_user_ingredients
from baseline_retrieval import tfidf_model, jaccard_model, minilm_model, initialise_minilm, recipes_dataset_normalised

# settings - same validation source split as the evaluations
split_seed = 34
validation_seed = 83
number_of_queries = 50
candidate_k = 100
runtime_repeats = 3
ingredient_counts_to_test = [3, 4, 5, 6]

models = {"TF-IDF": tfidf_model, "Jaccard": jaccard_model, "MiniLM": minilm_model}

# import shared random and manual query generation
from evaluation_queries import make_random_queries, make_manual_queries, MIN_SOURCE_INGREDIENTS

c:\Users\preen\Documents\AIML339 - FINAL\AIML339 - FINAL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# CREATE VALIDATION SOURCE SET

# use the same split as evaluations 1-4
# generate 50 source recipes with nested ingredient queries using seeds 34 and 83
random_queries = make_random_queries(recipes_dataset_normalised, partition="validation", query_lengths=ingredient_counts_to_test, number_of_queries=number_of_queries, split_seed=split_seed, query_seed=validation_seed, min_source_ingredients=MIN_SOURCE_INGREDIENTS)

# create 20 fixed manual queries - 5 groups with 3, 4, 5 and 6 ingredients
manual_queries = make_manual_queries()

# use 3 ingredients for the runtime test
runtime_queries = random_queries[random_queries["query_length"] == 3]["query"].tolist()
print("validation queries:", len(runtime_queries))
print("manual queries:", len(manual_queries))

validation queries: 50
manual queries: 20


## Baseline retrieval runtime


Measure per-query retrieval for the same 50 queries and 100 candidates. MiniLM embedding preparation is measured separately from its query retrieval. The first run of each method is a warm-up; the reported values exclude this warm-up.

In [12]:
# MEASURE ONE-TIME MINILM PREPARATION

# if embeddings are already cached, this will only measure the cached call
import baseline_retrieval as baseline_module  # import the retrieval module to inspect the MiniLM cache state
# check if recipe embeddings have already been loaded into memory
minilm_already_loaded = baseline_module.recipe_embeddings is not None
# start the timer before MiniLM setup begins 
#initialise MiniLM, build embedding cache if unabailable, then calculate how long preperation step took 
start = time.perf_counter()
initialise_minilm()
minilm_preparation_seconds = time.perf_counter() - start

# if MiniLM already loaded, clarify 
print("miniLM already loaded:", minilm_already_loaded)
# print time
print("miniLM preparation time (s):", round(minilm_preparation_seconds, 4))


miniLM already loaded: True
miniLM preparation time (s): 0.0007


In [13]:
# BASELINE RETRIEVAL RUNTIME

# empty list to collect runtimes
runtime_results = []

# iterate over baseline retrieval models
for model_name, model_function in models.items():
    # warm up model once before measuring to reduce first-call overhead
    model_function(runtime_queries[0], top_k=candidate_k)

    # repeat retrieval process for each query multiple times to estimate stable runtime
    for repeat in range(runtime_repeats):
        # eval each query in validation set
        for query_number, user_input in enumerate(runtime_queries):
            # time a single model call for this query
            start = time.perf_counter()
            recommendations = model_function(user_input, top_k=candidate_k)
            elapsed = time.perf_counter() - start

            # store information to list
            runtime_results.append({"model": model_name, "repeat": repeat + 1, "query_number": query_number + 1, "seconds": elapsed, "returned": len(recommendations)})

# convert to dataframe for summary analysis
runtime_results = pd.DataFrame(runtime_results)

# aggregate statistics across all repeats and queries
runtime_summary = runtime_results.groupby("model")["seconds"].agg(["mean", "std", "median", "min", "max"]).round(5)



## Different numbers of user ingredients

Use same 50 validation recipes for each input size. Calculate how often query ingredients appear in the top 5 recipes, average query coverage + missing recipe ingredients. 
Repeat with 20 fixed manual queries.

In [5]:
# GENERATE QUERIES WITH DIFFERENT INGREDIENT COUNTS

# random queries regenerated using the same seeds and source recipes
ingredient_count_queries = random_queries.rename(columns={"query_length": "ingredient_count"}).copy()
print("total queries:", len(ingredient_count_queries))

total queries: 200


In [19]:
# EVALUATE BASELINE INPUT ROBUSTNESS

# evaluate the random and manual queries using the same metrics
def evaluate_baseline_queries(query_table, experiment):
    # empty list to collect results
    robustness_results = []

    # iterate over each query
    for _, query_row in query_table.iterrows():
        # get the query and normalise its ingredients
        user_input = query_row["query"]
        user_set = set(normalise_user_ingredients(user_input))

        # evaluate the query using each retrieval model
        for model_name, model_function in models.items():
            # retrieve the top 5 recipe recommendations
            recommendations = model_function(user_input, top_k=5)

            # convert each recipe's ingredients to set
            recipe_sets = [set(ingredients) for ingredients in recommendations["processing_ingredients"]]

            # check whether each recipe contains all user ingredients
            full_matches = [user_set.issubset(recipe_set) for recipe_set in recipe_sets]

            # calculate proportion of user ingredients found in each recipe
            coverage = [len(user_set & recipe_set) / len(user_set) if user_set else 0 for recipe_set in recipe_sets]

            # count how many recipe ingredients not requested by user
            missing = [len(recipe_set - user_set) for recipe_set in recipe_sets]

            # store query details and average metrics for chosen model
            robustness_results.append({
                "experiment": experiment,
                "query_id": query_row["query_id"],
                "group": query_row.get("group", "random"),
                "query_number": query_row.get("query_number", None),
                "source_index": query_row.get("source_index", None),
                "query_length": query_row["query_length"],
                "query": user_input,
                "split_seed": query_row.get("split_seed", None),
                "query_seed": query_row.get("query_seed", None),
                "model": model_name,
                "precision_at_5": np.mean(full_matches),
                "coverage": np.mean(coverage),
                "missing": np.mean(missing)
            })

    # return results as dataframe
    return pd.DataFrame(robustness_results)

# evaluate random queries
random_results = evaluate_baseline_queries(random_queries, "random")

# evaluate manual queries
manual_results = evaluate_baseline_queries(manual_queries, "manual")

# calculate mean and standard deviation for each model and random query length
random_summary = random_results.groupby(["model", "query_length"])[["precision_at_5", "coverage", "missing"]].agg(["mean", "std"]).round(4)

# flatten column names after calculating summary statistics
random_summary.columns = ["_".join(column) for column in random_summary.columns]

# reset index to make grouped columns part of dataframe
random_summary = random_summary.reset_index()

# calculate mean metrics for each model, manual query group and query length
manual_summary = manual_results.groupby(["model", "group", "query_length"])[["precision_at_5", "coverage", "missing"]].mean().round(4).reset_index()


## Synonym + unknown-ingredient checks

Check if normalisation behaving as expected, if synonym matching is working + unknown terms are handled

In [18]:
# CHECK SYNONYM NORMALISATION

# define ingredient pairs that should normalise to same ingredient
synonym_pairs = [("scallion", "green onion"), ("spring onion", "green onion"), ("capsicum", "bell pepper"), ("icing sugar", "powdered sugar")]

# empty list to collect synonym normalisation results
synonym_results = []

# normalise each ingredient pair
for first, second in synonym_pairs:
    first_normalised = normalise_user_ingredients(first)
    second_normalised = normalise_user_ingredients(second)

    # store normalised ingredients and whether they match
    synonym_results.append({"first": first, "second": second, "first_normalised": str(first_normalised), "second_normalised": str(second_normalised), "same_normalisation": first_normalised == second_normalised})

# convert results to df
synonym_results = pd.DataFrame(synonym_results)


In [17]:
# CHECK IF SYNONYMS RETURN THE SAME RECIPES

# empty list to collect synonym retrieval results
synonym_retrieval_results = []

# check each synonym pair using each retrieval model
for first, second in synonym_pairs:
    for model_name, model_function in models.items():
        # retrieve top 5 recipes for each synonym
        first_results = model_function(first, top_k=5)
        second_results = model_function(second, top_k=5)

        # store whether both synonyms return same recipes in the same order
        synonym_retrieval_results.append({"model": model_name, "first": first, "second": second, "same_top_5_order": first_results.index.tolist() == second_results.index.tolist()})

# convert results to dataframe
synonym_retrieval_results = pd.DataFrame(synonym_retrieval_results)



In [10]:
# SAVE BASELINE ADDITIONAL TEST RESULTS

# save random and manual results - queries are regenerated using the fixed seeds
random_results.to_csv("baseline_randomquery.csv", index=False)
random_summary.to_csv("baseline_randomquery_summary.csv", index=False)
manual_results.to_csv("baseline_manual_userqueries.csv", index=False)
manual_summary.to_csv("baseline_manual_userqueries_summary.csv", index=False)

runtime_results.to_csv("baseline_runtime_results.csv", index=False)
runtime_summary.to_csv("baseline_runtime_summary.csv")
synonym_results.to_csv("baseline_synonym_normalisation.csv", index=False)
synonym_retrieval_results.to_csv("baseline_synonym_retrieval.csv", index=False)
unknown_results.to_csv("baseline_unknown_ingredients.csv", index=False)
print("baseline additional tests saved")

baseline additional tests saved
